# 004 — Unsupervised Baselines (PCC, HBOS, iForest, KNN)

## What's different from GlobalSTD (notebook 003)

- **Multivariate, not per-channel.** GlobalSTD thresholds each of the 6
  lightweight channels independently, then combines with OR. These four
  algorithms take all 6 channels as one feature vector per timestamp and
  output a single anomaly score directly — no combination step needed.
- **Unsupervised, not semi-supervised.** GlobalSTD's mean/std came from
  *nominal-only* training points. These train on the training set as-is,
  anomalies included (paper, Section 3.2: *"each algorithm is first
  initialized only on the training set — this includes calculating
  contamination levels... and then applied to the test set"*).
- **Standardization is used for the first time.** Deferred all the way back
  in notebook 002 for exactly this reason — the paper standardizes each
  channel using nominal-training mean/std (Section 3.3) before feeding it to
  these algorithms. GlobalSTD didn't need this since it computes its own
  mean/std internally; these four do.
- **Still no windowing.** Same reasoning as before — these operate on single
  timestamps, not sequences.

## What's deferred

**Windowed iForest** (Table 15 lists it separately, `window_size=17`) — it's
unsupervised too, no window-labeling risk, but scoped out to keep this
notebook to one clear addition (multivariate + standardization) at a time.
Good candidate for a short follow-up once these four are validated.

## Implementation choice: PyOD, not from scratch

Table 15's exact hyperparameters (`n_bins`, `bootstrap`, `n_neighbors`, etc.)
map directly onto PyOD's `PCA`, `HBOS`, `IForest`, and `KNN` classes —
confirmed by checking their constructor signatures directly. Reusing a
well-tested library here follows the same reasoning as reusing the reference
metrics code: less surface area for a from-scratch bug to hide in.

**One easy-to-miss parameter, caught before it became a silent bug:** Table 15
lists `max_samples: None` for iForest. PyOD/sklearn's current `IsolationForest`
rejects a literal `None` — the modern equivalent (each tree trained on *all*
training samples, not a 256-sample subsample) is `max_samples=1.0`. Verified
directly: `max_samples=1.0` on 1000 samples gives each tree exactly 1000
samples; PyOD's own default (`'auto'`) caps every tree at 256, which would
have been a very different (and non-paper-matching) algorithm.


## 0. Imports

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import portion as P
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "portion"])
    import portion as P

try:
    from pyod.models.pca import PCA
    from pyod.models.hbos import HBOS
    from pyod.models.iforest import IForest
    from pyod.models.knn import KNN
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyod"])
    from pyod.models.pca import PCA
    from pyod.models.hbos import HBOS
    from pyod.models.iforest import IForest
    from pyod.models.knn import KNN

print(f"pandas version: {pd.__version__}")


pandas version: 2.2.2


## 0.1 Mount Google Drive (same pattern as notebooks 002/003)

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path("/content/drive/MyDrive/BeaconProject")
    IN_COLAB = True
except ImportError:
    print("Not running in Colab — skipping Drive mount, falling back to a local path.")
    DRIVE_ROOT = Path("data/raw_drive_fallback")
    IN_COLAB = False

print(f"IN_COLAB = {IN_COLAB}")
print(f"DRIVE_ROOT = {DRIVE_ROOT}")


Mounted at /content/drive
IN_COLAB = True
DRIVE_ROOT = /content/drive/MyDrive/BeaconProject


## 1. Config

Table 4 numbers below are the lightweight-subset rows for all four algorithms,
both missions. `<0.001` in the paper is shown as `0.001` here (an upper bound,
not an exact value — worth remembering when reading the comparison later).


In [3]:
MISSION_CONFIG = {
    "ESA-Mission1": {
        "lightweight_channels": [f"channel_{i}" for i in range(41, 47)],
        "test_data_split": "2007-01-01",
        "resampling_rule": pd.Timedelta(seconds=30),
        "table4": {
            "PCC": {"precision": 0.001, "recall": 0.554, "f0.5": 0.001},
            "HBOS": {"precision": 0.001, "recall": 0.585, "f0.5": 0.001},
            "iForest": {"precision": 0.001, "recall": 0.585, "f0.5": 0.001},
            "KNN": {"precision": 0.001, "recall": 0.754, "f0.5": 0.001},
        },
    },
    "ESA-Mission2": {
        "lightweight_channels": [f"channel_{i}" for i in range(18, 29)],
        "test_data_split": "2001-10-01",
        "resampling_rule": pd.Timedelta(seconds=18),
        "table4": {
            "PCC": {"precision": 0.029, "recall": 1.000, "f0.5": 0.036},
            "HBOS": {"precision": 0.055, "recall": 0.911, "f0.5": 0.068},
            "iForest": {"precision": 0.557, "recall": 0.974, "f0.5": 0.609},
            "KNN": {"precision": 0.000, "recall": 1.000, "f0.5": 0.001},
        },
    },
}

# --- EDIT THIS ONE LINE to switch missions ---
ACTIVE_MISSION = "ESA-Mission1"

CFG = MISSION_CONFIG[ACTIVE_MISSION]
TARGET_CHANNELS = CFG["lightweight_channels"]
RESAMPLING_RULE = CFG["resampling_rule"]

RAW_DATA_ROOT = DRIVE_ROOT / ACTIVE_MISSION / ACTIVE_MISSION
PREPROCESSED_DIR = DRIVE_ROOT / "preprocessed" / ACTIVE_MISSION
PREDICTIONS_DIR = DRIVE_ROOT / "predictions" / ACTIVE_MISSION
PREDICTIONS_PATH = PREDICTIONS_DIR / "unsupervised_baselines.csv"

BETA = 0.5

print(f"Active mission: {ACTIVE_MISSION}")
print(f"Target channels: {TARGET_CHANNELS}")


Active mission: ESA-Mission1
Target channels: ['channel_41', 'channel_42', 'channel_43', 'channel_44', 'channel_45', 'channel_46']


## 2. Load preprocessed train/test (notebook 002's output)

In [4]:
anomaly_cols = [f"is_anomaly_{ch}" for ch in TARGET_CHANNELS]
dtypes = {ch: np.float64 for ch in TARGET_CHANNELS}
dtypes.update({c: np.uint8 for c in anomaly_cols})

train_df = pd.read_csv(PREPROCESSED_DIR / "train.csv", index_col="timestamp", parse_dates=True, dtype=dtypes)
test_df = pd.read_csv(PREPROCESSED_DIR / "test.csv", index_col="timestamp", parse_dates=True, dtype=dtypes)

print(f"train: {train_df.shape}, test: {test_df.shape}")


train: (7099200, 12), test: (7364161, 12)


## 3. Load raw event annotations (same tz-fix and Category merge as notebook 003)

In [5]:
labels_df = pd.read_csv(RAW_DATA_ROOT / "labels.csv", parse_dates=["StartTime", "EndTime"])
anomaly_types_df = pd.read_csv(RAW_DATA_ROOT / "anomaly_types.csv")

if labels_df["StartTime"].dt.tz is not None:
    labels_df["StartTime"] = labels_df["StartTime"].dt.tz_localize(None)
if labels_df["EndTime"].dt.tz is not None:
    labels_df["EndTime"] = labels_df["EndTime"].dt.tz_localize(None)

labels_df = labels_df.merge(anomaly_types_df[["ID", "Category"]], on="ID", how="left")

y_true = labels_df[labels_df["Channel"].isin(TARGET_CHANNELS)].copy()
y_true = y_true[y_true["StartTime"] >= pd.to_datetime(CFG["test_data_split"])]
print(f"Total events in the test period affecting target channels: {y_true['ID'].nunique()}")


Total events in the test period affecting target channels: 65


## 4. The metric (same validated scorer as notebook 003)

Copied rather than imported, so this notebook runs standalone in a fresh
Colab session without depending on 003 having been run first. Same three
checks as before — quick re-verification, not re-litigation.


In [6]:
def convert_time_series_to_events(vector) -> P.Interval:
    vector = np.asarray(vector)

    def find_runs(x):
        x = np.asanyarray(x)
        n = x.shape[0]
        if n == 0:
            return np.array([]), np.array([]), np.array([])
        loc_run_start = np.empty(n, dtype=bool)
        loc_run_start[0] = True
        np.not_equal(x[:-1], x[1:], out=loc_run_start[1:])
        run_starts = np.nonzero(loc_run_start)[0]
        run_values = x[loc_run_start]
        run_lengths = np.diff(np.append(run_starts, n))
        run_ends = run_starts + run_lengths
        return np.stack((run_starts[run_values > 0], run_ends[run_values > 0])).transpose()

    non_zero_runs = find_runs(vector[..., 1])
    events = []
    n = len(vector)
    for x, y in non_zero_runs:
        if y == n:
            events.append(P.closed(vector[..., 0][x], vector[..., 0][y - 1]))
        else:
            events.append(P.closedopen(vector[..., 0][x], vector[..., 0][y]))
    return P.Interval(*events)


class EventWiseScorer:
    NANOSECONDS_IN_SECOND = 1e9

    def __init__(self, betas=1.0, select_labels=None, full_range=None):
        self._betas = np.atleast_1d(betas)
        self.full_range = full_range
        if select_labels is None or len(select_labels) == 0:
            self.selected_labels = {}
        else:
            self.selected_labels = {c: np.atleast_1d(v) for c, v in select_labels.items()}

    def score(self, y_true: pd.DataFrame, y_pred) -> dict:
        y_pred = np.asarray(y_pred)

        if self.full_range is None:
            self.full_range = (min(y_true["StartTime"].min(), min(y_pred[..., 0])),
                               max(y_true["EndTime"].max(), max(y_pred[..., 0])))
        if y_pred[0, 0] > self.full_range[0]:
            y_pred = np.array([np.array([self.full_range[0], y_pred[0, 1]]), *y_pred])
        if y_pred[-1, 0] < self.full_range[1]:
            y_pred = np.array([*y_pred, np.array([self.full_range[1], y_pred[-1, 1]])])

        events_pred = convert_time_series_to_events(y_pred)

        filtered_y_true = y_true.copy()
        for col, val in self.selected_labels.items():
            filtered_y_true = filtered_y_true[filtered_y_true[col].isin(val)]

        true_positives = 0
        false_negatives = 0
        redundant_detections = 0
        matched_events_pred = [False for _ in events_pred]

        for aid in filtered_y_true["ID"].unique():
            gt = filtered_y_true[filtered_y_true["ID"] == aid]
            gt_intervals = P.Interval(*[P.closed(*row) for _, row in gt[["StartTime", "EndTime"]].iterrows()])

            already_detected = [0 for _ in gt_intervals]
            at_least_one_detected = False
            for p, pred in enumerate(events_pred):
                if pred.upper < gt_intervals.lower or pred.lower > gt_intervals.upper:
                    continue
                intersections = [not (pred & g).empty for g in gt_intervals]
                if not any(intersections):
                    continue
                matched_events_pred[p] = True
                if not at_least_one_detected:
                    true_positives += 1
                    at_least_one_detected = True
                for i, val in enumerate(intersections):
                    if val:
                        already_detected[i] += 1

            for det in already_detected:
                if det > 1:
                    redundant_detections += (det - 1)
            if not at_least_one_detected:
                false_negatives += 1

        events_gt = P.Interval(*[P.closed(*row) for _, row in y_true[["StartTime", "EndTime"]].iterrows()])
        false_positives = sum(
            1 for pred, matched in zip(events_pred, matched_events_pred)
            if not matched and (pred & events_gt).empty
        )

        divider = true_positives + false_positives
        precision = 0.0 if divider == 0 else true_positives / divider

        divider = true_positives + redundant_detections
        alarming_precision = 0.0 if divider == 0 else true_positives / divider

        if precision > 0:
            nominal_interval = P.closed(*self.full_range) - events_gt
            false_positives_interval = nominal_interval & events_pred
            nominal_seconds = sum((iv.upper - iv.lower).value / self.NANOSECONDS_IN_SECOND for iv in nominal_interval)
            fp_seconds = sum((iv.upper - iv.lower).value / self.NANOSECONDS_IN_SECOND for iv in false_positives_interval)
            tnr = 1 - fp_seconds / nominal_seconds if nominal_seconds > 0 else 1.0
            precision *= tnr

        divider = true_positives + false_negatives
        recall = 0.0 if divider == 0 else true_positives / divider

        result = {
            "TP": true_positives, "FP": false_positives, "FN": false_negatives,
            "alarming_precision": alarming_precision,
            "EW_precision": precision, "EW_recall": recall,
        }
        for b in self._betas:
            divider = b ** 2 * precision + recall
            result[f"EW_F_{b:.2f}"] = 0.0 if divider == 0 else ((1 + b ** 2) * precision * recall) / divider
        return result


# --- quick re-verification against the reference's own worked example ---
_full_range = (pd.to_datetime("2015-01-01"), pd.to_datetime("2015-01-15"))
_y_true = pd.DataFrame([
    ["id_0", pd.to_datetime("2015-01-01"), pd.to_datetime("2015-01-02"), "Multivariate", "Point"],
    ["id_1", pd.to_datetime("2015-01-04"), pd.to_datetime("2015-01-05"), "Univariate", "Subsequence"],
    ["id_2", pd.to_datetime("2015-01-07"), pd.to_datetime("2015-01-08"), "Multivariate", "Subsequence"],
], columns=["ID", "StartTime", "EndTime", "Dimensionality", "Length"])
_y_pred = [[pd.to_datetime("2015-01-01"), 0], [pd.to_datetime("2015-01-04"), 1], [pd.to_datetime("2015-01-09"), 0]]
_scorer = EventWiseScorer(betas=0.5, full_range=_full_range,
                          select_labels={"Dimensionality": "Multivariate", "Length": "Subsequence"})
_result = _scorer.score(_y_true, _y_pred)
assert abs(_result["EW_precision"] - 0.7272727272727273) < 1e-9
assert abs(_result["EW_F_0.50"] - 0.7692307692307693) < 1e-9
print("PASS: scorer re-verified against the reference implementation's own worked example.")


PASS: scorer re-verified against the reference implementation's own worked example.


## 5. Standardization (paper Section 3.3)

Used for the first time in this project. Per-channel mean/std computed from
*nominal-labeled training points only* — same nominal-only principle as
GlobalSTD's threshold, just applied as a transform (z-score) instead of a
threshold this time. Constant channels get a std of 1 instead of 0, to avoid
dividing by zero (same guard as GlobalSTD used).


In [7]:
def fit_standardization(train_df: pd.DataFrame, target_channels: list) -> tuple[dict, dict]:
    means, stds = {}, {}
    for ch in target_channels:
        label_col = f"is_anomaly_{ch}"
        nominal_values = train_df.loc[train_df[label_col] == 0, ch]
        means[ch] = nominal_values.mean()
        std = nominal_values.std()
        stds[ch] = std if std > 0 else 1.0
    return means, stds


def apply_standardization(df: pd.DataFrame, target_channels: list, means: dict, stds: dict) -> pd.DataFrame:
    out = df.copy()
    for ch in target_channels:
        out[ch] = (out[ch] - means[ch]) / stds[ch]
    return out


std_means, std_stds = fit_standardization(train_df, TARGET_CHANNELS)
train_std = apply_standardization(train_df, TARGET_CHANNELS, std_means, std_stds)
test_std = apply_standardization(test_df, TARGET_CHANNELS, std_means, std_stds)

X_train = train_std[TARGET_CHANNELS].values
X_test = test_std[TARGET_CHANNELS].values
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Post-standardization train means (should be ~0): {X_train.mean(axis=0).round(3)}")


X_train: (7099200, 6), X_test: (7364161, 6)
Post-standardization train means (should be ~0): [ 0.028  0.02  -0.028 -0.105  0.028  0.023]


## 6. Contamination

The paper's algorithms "calculate contamination levels... during training"
(Section 3.2) rather than use an arbitrary fixed guess. Since our training
data carries real labels, we can do this directly: the actual fraction of
training timestamps where *any* target channel was Anomaly or Rare Event
(matching the same combined, multi-channel view used everywhere else).
Clamped to PyOD's accepted range.


In [8]:
anomaly_label_cols = [f"is_anomaly_{ch}" for ch in TARGET_CHANNELS]
is_anomalous_row = (train_df[anomaly_label_cols].isin([1, 2])).any(axis=1)
contamination = is_anomalous_row.mean()
contamination = float(np.clip(contamination, 0.001, 0.5))
print(f"Estimated contamination from training labels: {contamination:.6f}")


Estimated contamination from training labels: 0.012504


## 7. The four models

Exact hyperparameters from Table 15. `random_state=42` everywhere the paper
specifies it, for reproducibility.

**Runtime expectations, from the paper's own Appendix D.8** (Mission1
lightweight, for calibration — yours will differ with Colab's CPU, but same
order of magnitude): PCC ~90s train / ~124s predict, HBOS ~110s / ~135s,
iForest ~655s / ~393s, **KNN ~3844s (~1hr) / ~1233s (~20min)**. KNN is the
long pole by a wide margin — worth starting it and doing something else while
it runs, not worth panicking if it takes a while.


In [9]:
import time

fitted_models = {}
predictions_raw = {}

def save_predictions_to_drive():
    PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
    predictions_df = pd.DataFrame(predictions_raw, index=test_df.index)
    predictions_df.index.name = "timestamp"
    predictions_df.to_csv(PREDICTIONS_PATH)
    print(f"Saved predictions for {list(predictions_raw.keys())} to {PREDICTIONS_PATH}")

if PREDICTIONS_PATH.exists():
    _cached = pd.read_csv(PREDICTIONS_PATH, index_col="timestamp", parse_dates=True)
    for col in _cached.columns:
        predictions_raw[col] = _cached[col].values
    print(f"Loaded cached predictions from {PREDICTIONS_PATH}: {list(predictions_raw.keys())}")
    print("Any model listed above will be SKIPPED below -- delete its column from the "
          "cached CSV (or the whole file) to force a re-run.")
else:
    print(f"No cached predictions found at {PREDICTIONS_PATH} -- all four models will run fresh.")

No cached predictions found at /content/drive/MyDrive/BeaconProject/predictions/ESA-Mission1/unsupervised_baselines.csv -- all four models will run fresh.


### 7.1 PCC (PyOD `PCA`)

In [10]:
import time
if "PCC" in predictions_raw:
    print("PCC already loaded from cache -- skipping.")
else:
    _t0 = time.time()

    pcc = PCA(n_components=None, n_selected_components=None, svd_solver="auto",
            tol=0.0, whiten=False, random_state=42, standardization=False,  # already standardized ourselves
            contamination=contamination)
    pcc.fit(X_train)
    fitted_models["PCC"] = pcc
    print(f"PCC trained in {time.time() - _t0:.1f}s")

    _t0 = time.time()
    predictions_raw["PCC"] = pcc.predict(X_test)
    print(f"PCC predicted in {time.time() - _t0:.1f}s")
    print(f"Flagged {predictions_raw['PCC'].mean() * 100:.4f}% of test timestamps")

    save_predictions_to_drive()

PCC trained in 0.9s
PCC predicted in 0.7s
Flagged 8.1234% of test timestamps
Saved predictions for ['PCC'] to /content/drive/MyDrive/BeaconProject/predictions/ESA-Mission1/unsupervised_baselines.csv


### 7.2 HBOS

In [11]:
if "HBOS" in predictions_raw:
    print("HBOS already loaded from cache -- skipping.")
else:
    _t0 = time.time()

    hbos = HBOS(n_bins=50, alpha=0.1, tol=0.5, contamination=contamination)
    hbos.fit(X_train)
    fitted_models["HBOS"] = hbos
    print(f"HBOS trained in {time.time() - _t0:.1f}s")

    _t0 = time.time()
    predictions_raw["HBOS"] = hbos.predict(X_test)
    print(f"HBOS predicted in {time.time() - _t0:.1f}s")
    print(f"Flagged {predictions_raw['HBOS'].mean() * 100:.4f}% of test timestamps")
    save_predictions_to_drive()

HBOS trained in 4.7s
HBOS predicted in 1.1s
Flagged 3.0793% of test timestamps
Saved predictions for ['PCC', 'HBOS'] to /content/drive/MyDrive/BeaconProject/predictions/ESA-Mission1/unsupervised_baselines.csv


### 7.3 iForest

`max_samples=1.0` — see the note in the intro: this is the modern equivalent
of Table 15's `max_samples: None` (every tree trained on the full training
set, not PyOD's default 256-sample cap per tree).


In [12]:
if "iForest" in predictions_raw:
    print("iForest already loaded from cache -- skipping.")
else:
    _t0 = time.time()
    iforest = IForest(n_estimators=100, bootstrap=False, max_features=1.0,
                      max_samples=1.0, random_state=42, contamination=contamination)
    iforest.fit(X_train)
    fitted_models["iForest"] = iforest
    print(f"iForest trained in {time.time() - _t0:.1f}s")

    _t0 = time.time()
    predictions_raw["iForest"] = iforest.predict(X_test)
    print(f"iForest predicted in {time.time() - _t0:.1f}s")
    print(f"Flagged {predictions_raw['iForest'].mean() * 100:.4f}% of test timestamps")
    save_predictions_to_drive()

iForest trained in 362.0s
iForest predicted in 64.6s
Flagged 28.8163% of test timestamps
Saved predictions for ['PCC', 'HBOS', 'iForest'] to /content/drive/MyDrive/BeaconProject/predictions/ESA-Mission1/unsupervised_baselines.csv


### 7.4 KNN

**The slow one.** If this is taking a genuinely long time and you want to
sanity-check the rest of the pipeline first, skip this cell and come back —
every later cell handles a missing `"KNN"` entry in `predictions_raw`
gracefully (just run the scoring cells for the three you have).


In [13]:
if "KNN" in predictions_raw:
    print("KNN already loaded from cache -- skipping.")
else:
    _t0 = time.time()

    knn = KNN(n_neighbors=5, method="largest", leaf_size=30, p=2, contamination=contamination)
    knn.fit(X_train)
    fitted_models["KNN"] = knn
    print(f"KNN trained in {time.time() - _t0:.1f}s")

    _t0 = time.time()
    predictions_raw["KNN"] = knn.predict(X_test)
    print(f"KNN predicted in {time.time() - _t0:.1f}s")
    print(f"Flagged {predictions_raw['KNN'].mean() * 100:.4f}% of test timestamps")
    save_predictions_to_drive()


KNN trained in 685.8s
KNN predicted in 549.2s
Flagged 26.4198% of test timestamps
Saved predictions for ['PCC', 'HBOS', 'iForest', 'KNN'] to /content/drive/MyDrive/BeaconProject/predictions/ESA-Mission1/unsupervised_baselines.csv


## 8. Score against real events

Same pattern as notebook 003: `select_labels` applied at scoring time (not by
pre-filtering), strict scoring for the paper-comparable number, tolerant
scoring (GT padded by one resampling step) alongside it to account for the
same forward-shift artifact documented there — these operate on the same
30-second grid, so the same mechanism applies equally here.


In [14]:
full_range = (test_df.index.min(), test_df.index.max())

results_strict = {}
results_tolerant = {}

for name, pred in predictions_raw.items():
    y_pred_pairs = list(zip(test_df.index, pred.astype(np.uint8)))

    scorer = EventWiseScorer(betas=BETA, full_range=full_range,
                             select_labels={"Category": ["Rare Event", "Anomaly"]})
    results_strict[name] = scorer.score(y_true, y_pred_pairs)

    y_true_padded = y_true.copy()
    y_true_padded["StartTime"] = y_true_padded["StartTime"] - RESAMPLING_RULE
    y_true_padded["EndTime"] = y_true_padded["EndTime"] + RESAMPLING_RULE
    scorer_tol = EventWiseScorer(betas=BETA, full_range=full_range,`2222
                                 select_labels={"Category": ["Rare Event", "Anomaly"]})
    results_tolerant[name] = scorer_tol.score(y_true_padded, y_pred_pairs)

    print(f"{name}: strict={results_strict[name]}")


PCC: strict={'TP': 39, 'FP': 347177, 'FN': 26, 'alarming_precision': 0.038461538461538464, 'EW_precision': 0.00010410096944289431, 'EW_recall': 0.6, 'EW_F_0.50': np.float64(0.0001301205677714306)}
HBOS: strict={'TP': 27, 'FP': 104879, 'FN': 38, 'alarming_precision': 0.046232876712328765, 'EW_precision': 0.0002516814865246269, 'EW_recall': 0.4153846153846154, 'EW_F_0.50': np.float64(0.0003145542110664948)}
iForest: strict={'TP': 46, 'FP': 560044, 'FN': 19, 'alarming_precision': 0.03514132925897632, 'EW_precision': 5.881838774286892e-05, 'EW_recall': 0.7076923076923077, 'EW_F_0.50': np.float64(7.352145703248906e-05)}
KNN: strict={'TP': 49, 'FP': 938273, 'FN': 16, 'alarming_precision': 0.01325040562466198, 'EW_precision': 3.869264429636539e-05, 'EW_recall': 0.7538461538461538, 'EW_F_0.50': np.float64(4.836518476077374e-05)}


## 9. Compare against the paper's Table 4

Same calibration note as notebook 003 applies: strict is the methodologically
fair comparison point, tolerant is a diagnostic upper bound that confirms
whether the forward-shift artifact is contributing to any gap here too. These
four algorithms are less threshold-sensitive than GlobalSTD (continuous
scores + a contamination-based cutoff, not one hard mean±std line), so
agreement here is a cleaner signal about the pipeline than GlobalSTD could
give — if these land close, that's strong independent confirmation.


In [15]:
print(f"{'Algorithm':<10} {'Metric':<10} {'Strict':>10} {'Tolerant':>10} {'Paper (Table 4)':>16}")
print("-" * 72)
for name in ["PCC", "HBOS", "iForest", "KNN"]:
    if name not in results_strict:
        print(f"{name}: skipped (no predictions)")
        continue
    paper = CFG["table4"][name]
    strict = results_strict[name]
    tol = results_tolerant[name]
    rows = [
        ("precision", strict["EW_precision"], tol["EW_precision"], paper["precision"]),
        ("recall", strict["EW_recall"], tol["EW_recall"], paper["recall"]),
        (f"F_{BETA:.2f}", strict[f"EW_F_{BETA:.2f}"], tol[f"EW_F_{BETA:.2f}"], paper["f0.5"]),
    ]
    for metric_name, strict_val, tol_val, paper_val in rows:
        print(f"{name:<10} {metric_name:<10} {strict_val:>10.4f} {tol_val:>10.4f} {paper_val:>16.4f}")
    print()


Algorithm  Metric         Strict   Tolerant  Paper (Table 4)
------------------------------------------------------------------------
PCC        precision      0.0001     0.0002           0.0010
PCC        recall         0.6000     0.8923           0.5540
PCC        F_0.50         0.0001     0.0002           0.0010

HBOS       precision      0.0003     0.0004           0.0010
HBOS       recall         0.4154     0.7231           0.5850
HBOS       F_0.50         0.0003     0.0005           0.0010

iForest    precision      0.0001     0.0001           0.0010
iForest    recall         0.7077     0.9385           0.5850
iForest    F_0.50         0.0001     0.0001           0.0010

KNN        precision      0.0000     0.0001           0.0010
KNN        recall         0.7538     1.0000           0.7540
KNN        F_0.50         0.0000     0.0001           0.0010



## 9.1 Per-event breakdown

Same diagnostic shape as notebook 003 — useful if any algorithm's numbers
look off and you want to see *which* events specifically are driving it,
split by category (recall notebook 003's finding: Rare Events are
structurally hard for simple methods on this channel subset, so a low
combined recall dragged down mostly by Rare Events isn't necessarily a bug).


In [16]:
for cat in ["Anomaly", "Rare Event"]:
    print(f"\n=== {cat} only ===")
    for name, pred in predictions_raw.items():
        y_pred_pairs = list(zip(test_df.index, pred.astype(np.uint8)))
        scorer = EventWiseScorer(betas=BETA, full_range=full_range, select_labels={"Category": [cat]})
        r = scorer.score(y_true, y_pred_pairs)
        print(f"  {name:<10}: precision={r['EW_precision']:.4f}, recall={r['EW_recall']:.4f}, "
              f"F_{BETA:.2f}={r[f'EW_F_{BETA:.2f}']:.4f}")



=== Anomaly only ===
  PCC       : precision=0.0000, recall=0.3103, F_0.50=0.0000
  HBOS      : precision=0.0001, recall=0.2759, F_0.50=0.0001
  iForest   : precision=0.0000, recall=0.4828, F_0.50=0.0000
  KNN       : precision=0.0000, recall=0.4483, F_0.50=0.0000

=== Rare Event only ===
  PCC       : precision=0.0001, recall=0.8333, F_0.50=0.0001
  HBOS      : precision=0.0002, recall=0.5278, F_0.50=0.0002
  iForest   : precision=0.0000, recall=0.8889, F_0.50=0.0001
  KNN       : precision=0.0000, recall=1.0000, F_0.50=0.0000
